# Instructions
The setup for this lesson is a bit more involved than the previous. Please follow these instructions carefully:

## Change Runtime
First we want to connect Google Colab to a GPU. A GPU (aka a Graphics Processing Unit) is a device which allows a computer to quickly do the math required to run and train neural networks.

1. Click the Runtime > Change runtime type menu option at the top of the page:
   ![Screenshot of change runtime type menu](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-change-runtime-type-menu.png?raw=true)
2. Then select "T4 GPU" under the hardware accelerator section and click "Save"
   ![Change runtime type popup screenshot](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-change-runtime-type-popup.png?raw=true)  
   Sometimes Google Colab does not have any T4 GPUs available. In this case you will see a message like this:  
   ![Error message about no GPUs](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-no-gpu-runtime-popup.png?raw=true)  
   Although not ideal you can still continue with the lesson. Some steps just might take a little longer.

## Install Packages
We need to install some additional packages to support our neural network for this lesson.

1. Run the code in the cell below
2. This will take some time (1-3 minutes)
3. When it completes you will see a pop-up telling you to restart the session. Click the "Restart Session" button:  
   ![Restart session pop-up](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/post-pip-install-restart-session.png?raw=true)

In [ ]:
!pip install executorch torch torchvision

## Login to Google Drive
This lesson will use photos which you upload to your Google Drive to train a neural network. In order to use these photos you must login to Google Drive.  

Run the code in the cell directly below. Right after you do so a window will pop up asking you to connect to Google Drive.


1. Click the "Connect to Google Drive" button  
  ![Connect with google drive popup](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-request-drive-access-popup.png?raw=true)  
2. Click the "Continue" button
  ![Sign in start page](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-drive-desktop-sign-in.png?raw=true)  
3. Click the "Select all" check box and then the "Continue" button
  ![Sign in permissions](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-drive-desktop-permissions.png?raw=true)

Now code in this notebook has access to your Google Drive files. We will just be reading images. But be careful if you decide to use code from the internet or from a LLM / chat bot as it will have access to all your Google Drive files.

_Run the code below, make sure to follow the instructions above_

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Final Python Setup
Finally run the cell below. This imports some helpers which we will need to make our neural network:

In [ ]:
from collections import defaultdict
import json

from ipywidgets import widgets
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import random_split, Subset, Dataset, DataLoader
from torchvision import datasets, transforms, models

# Image Classifiers
We are going to create a neural network which can recognize objects in photos. This type of neural network is called a "classifier". This is because the neural network _clasifies_ images based on the object in the image.

- Each object that an image classifier can recognize is called a "class"
- For each class image classifiers output a probability of the input image contains that class
  - Example: 80% chance image contains a duck, 20% chance it contains a llama

For example if we make an image classifier than can recognize llamas and ducks. The classes are:
  
- Ducks
- Llamas

Image classifiers can only recognize the classes you train them on. They will always output the probability of the image containing each class.

- This means if you show an image classifier something it isn't trained to recognize it will still try to classify it into its known objects
- With our llama or duck classifier if we show it a truck
  - The classifier will still try to determine the probability that the image contains a duck or a llama
  - It won't be able to say "Oh I don't know"

Image classifiers have a set number of
In this notebook we are going to create a neural network which is trained to recognize objects in images. You will upload photos to use as training data to help the neural network learn what these objects look like.

## Where Will We Run the Neural Network?
Our plan is to run this neural network on a Raspberry Pi. This Raspberry Pi is a very small computer. Even smaller than your cellphone, and even less powerful.

Because we are running a neural network on this device we need to be careful about the structure of our neural network.

- Not only do we need to make a neural network which is effective
- It also needs to be efficient enough that we can run it on a small computer

Luckily the machine learning community has spent a lot of time figuring out how to do this. Researchers have created several neural networks focused on image classification on mobile devices.

- We will be using a neural network named MobileNet 2
- It is meant to run on smartphones
- Designed to recognize what object is in an image (it is an image classifier)
- MobileNet 2 is a convolutional neural network



## Training MobileNet 2
MobileNet 2 is a pretty big neural network:

- 53 layers
- 3.4 million parameters
- ~5,000 pixels of image as input
- Has to determine what object is in those pixels

To train a model like this you need a large dataset:

- 1.28 million images
- Of 1,000 objects

Training a big model like this with such a large dataset takes days:

- With 8 large expensive GPUs
- Running non-stop
- Training would take 2-3 days

Because of this we run in to several problems:

- We don't have 8 large expensive GPUs
- We don't have multiple days to train our neural network

### Fine Tuning
There is a solution to our problem of training MobileNet 2. We can do a process called **fine tuning**.

#### What Does Training MobileNet 2 Do?
During the process of training MobileNet 2 what is the neural network learning?

- Sure it's learning how to identify the 1,000 objects in the training dataset
- But what else?
- It's learning how to differentiate between objects
- It learns to look at how sharp corners are
- Or to pay attention to outlines

During training MobileNet 2 is learning general skills that can be used to recognize any object.

- It's like how in math class you learn how to do math by practicing solving equations
- After a year of math you aren't just able to solve the equations you saw on your worksheets and tests
- You've built the skills required to solve similar equations

The same is true of a complex neural network like MobileNet 2. After training it's built up the skills required to recognize and differentiate between objects in general.

#### Fine Tuning MobileNet 2
We can take advantage of the fact that after training MobileNet 2 has the skills to look at and understand objects.

- Instead of starting our training process with random weights and biases
- We start the training process with the weights and biases from the fully trained MobileNet 2 model
  - This gives us a 2-3 day headstart on training
  - Now right off the bat our neural network has good enoughts weights and biases to correctly categorize 1,000 objects

We take the output layer of MobileNet 2 and replace it with our own.

- Instead of 1,000 output neurons
  - 1 for each object the model can recognize
  - Each outputing a number from 0.0 to 1.0
  - A percentage from 0% to 100% of how sure the model is that the image is of that object
- Our replacement output layer will have the number of neurons equal to the number of objects we want to recognize
  - So if we only want to categorize dogs versus cats
  - We have 2 output neurons
  - Each outputting 0.0 to 1.0 as a probability of how sure the model is that a dog or cat is in the image

Then during training we don't touch the weights or biases for any other layer. We only modify the weights and biases of our final output layer.

- This lets MobileNet 2 retain all of its skills that it built up for recognizing objects
- By only training our final layer we need a much smaller computer and much less time to train
- We don't have millions of photos of the objects we want to categorize

In reality, few people ever train a complex neural network from scratch. For the exact reasons we mentioned above. A lot of the time people using the process of fine tuning to repurpose existing models.

# Check In - MobileNet 2 Fine Tuning
We just learned about MobileNet 2 and the process of fine tuning. Here are the points you should understand:

- MobileNet 2 is a neural network made to run on not so powerful devices like cell phones
- It would take way too long and cost way too much money to train MobileNet 2 from scratch
- Fine tuning is the process of replacing the output layer of an existing neural network
- Fine tuning let's us repurpose an already trained neural network for our specific task

# Training Data
We are going to fine tune MobileNet 2 to recognize our own categories (aka classes) of objects. We will need around 70-100 photos of each item we want our neural network to recognize.

Look through your camera roll and see if you have 70-100 photos of at least 2 different categories of items. Good examples are:

- Pets
- Food
- Sunsets

If you don't have enough photos then don't worry, you can always find an item in the classroom and take a bunch of photos.

Make sure you pick 2-3 objects.

## Prepare Your Photos on Google Drive
We want to make a folder on Google Drive which holds our images for each type of object we want to our neural network to detect.

1. In a different tab open Google Drive
2. Find a good location to put our images
3. Create a folder named "Images Classifier Datasets"
4. Inside this folder create a sub-folder for each object we want to recognize
  - For example if you want your neural network to recognize cats and trains make 2 folders named "Cats" and another one named "Trains"
5. Upload your training images into each folder

For example your folders should look something like this:

![Google drive datasets folder structure](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/google-drive-folder-structure.png?raw=true)

_(But your folders should have the names of the objects you're trying to recognize)_

Inside each folder there should be 70-100 images of the object you want the neural network to recognize.

Now go back to this Google Colab tab in your browser.

1. On the left hand side click the "Files" icon to open the file explorer:  
   ![Google colab file explorer icon](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-file-explorer-icon.png?raw=true)
2. Find the "content" > "drive" folder:  
   ![/content/drive folder](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-file-explorer-content-drive.png?raw=true)
3. Then find your "Image Classifier Datasets" folder in this file explorer, hover over the right hand side and click on the 3 dots menu:
   ![Image classifier datasets 3 dots menu](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-file-explorer-dataset-folder.png?raw=true)  
4. In the menu click the "Copy path" option:  
   ![Image classifiers datasets copy path option](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-file-explorer-dataset-folder-copy-path.png?raw=true)  
   This will copy the location of your datasets folder onto your clipboard
5. Paste the location of your datasets folder in as the value of the `datasets_parent` variable in the code cell below. Then run the cell.

In [ ]:
#datasets_parent = "REPLACE ME WITH DATASETS PATH"
datasets_parent = "/content/drive/MyDrive/Work/Putny Nat Geo/Putney as Collab/Image Classifier Datasets"

## Loading Dataset in PyTorch
Now that we have our training data all organized into folders of images we need to tell PyTorch where it is.

- If you remember back to the previous lesson about neural networks you'll remember that we need to create class which inherits from `Dataset`
- Luckily organizing images by folder is a common pattern and PyTorch already has a class which inherits from `Dataset` to help us out: `ImageFolder`

Run the code below, this will create a dataset which PyTorch can use to load images for each of our classes.

In [ ]:
dataset = datasets.ImageFolder(datasets_parent)

This `ImageFolder` class is also nice because it knows the names of our classes, check the `dataset.classes` variable to see if it picked up your classes correctly:

_Run the code below_

In [ ]:
dataset.classes

Now that we have our dataset ready we can click the "Run all" button in Google Colab:

![Google colab run all button](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/run-all-button.png?raw=true)

If you ever upload or delete anything in your "Image Classifier Datasets" folder be sure to click this "Run all" button again.

**Important:** If you a popup asking to "Restart session" after clicking this button then click yes, and then click the "Run all" button **again**

- If it has been a long enough time Google Colab will make you re-run the "Connect Google Drive" and "Package Install" steps from the top of this notebook
- If you are asked to restart the session then do so
  - But then you must click the "Run all" button **again**
  


## Normalizing our Data
It is important to normalize your training data before training a neural network. This ensures the internals of the neurons function properly and training goes well.

In a previous lesson we made a neural network to predict if a day would have good weather based on the temperature, humidity, and cloud coverage. You may remember that we normalized the humidity and cloud coverage values to be within the range of 0.0 and 1.0.

- We need to do the same thing for the red, green, and blue pixel values in our image
- We need to take each pixel's red, green, and blue values and convert them to be within 0.0 and 1.0

### Normalizing Image Size
Since we are dealing with images we also need to make sure the size of our images are normalized.

- MobileNet 2 expects images to be 224 pixels wide and 224 pixels high
- We need to take our images and crop them to be this size

### Normalizing When Fine Tuning
Since we are fine tuning an existing model we need to take our normalization process one step further.

- We need to ensure the statistics properties of our training data matches the original training data used to train MobileNet 2
- By statistics properties we are talking about:
  - The average value of the red, green, and blue pixels across all our images in the dataset
  - The standard deviation of the red, green, and blue pixels across all our images in the dataset

> If you haven't heard about standard deviation don't worry too much.  
> It's just a measurement of how different pieces of data in our dataset are from each other

By making sure our training data has similar properties to the original MobileNet 2 training data we ensure that training will go smoothly and that the model will perform correctly.

MobileNet 2's training data had an average and standard deviation of:

| - | Red | Green | Blue |
| - | - | - | - |
| **Average** | $0.485$ | $0.456$ | $0.406$ |
| **Standard Deviation** | $0.229$ | $0.224$ | $0.225$ |

So in our training data we need to change the values of the red, green, and blue pixels of all our images so that:

- Values are normalized between 0.0 and 1.0
- Values match the averages and standard deviations of MobileNet 2's training data

# Check In - Normalizing Training Data
We just learned about normalizing training data when fine tuning MobileNet 2. Here are the key points you should understand:

- Just like our weather neural network, we want all values to be between 0.0 and 1.0
- We need to ensure our images are 224 pixels by 224 pixels in size
- We need to normalize the values for the amount of red, green, and blue in each pixel of each image
- The average and standard deviation of our training data must match that of the original MobileNet 2 training data
  - This is required because we are using the weights which were trained using this original data
  - For our fine tuning to work our training data needs to match

## Normalizing Input Data
The neural network is trained on images of a certain size, with certain properties (normalized 0.0 to 1.0, average, standard deviation).

When using our neural network later to identify images we also need to normalize these images before we give them as input. This is because all the weights and biases in our neural network are used to images with those properties. So for our neural network to work, our input images must also have these same properties.

## Normalizing Data with PyTorch
### Transforms
PyTorch provides helpers to normalize all our data as we just described. These helpers are called "transforms" since they transform the images:

- `transforms.Compose` is a class which lets you combine multiple normalization steps into one
- `transforms.Resize` is a class which resizes an image
- `transforms.CenterCrop` is a class which crops an image to a specific size
- `transforms.Normalize` is a class which ensures images have a certain average and standard deviation value for their red, green, and blue pixels

We can combine these classes to normalize our data.

PyTorch also provides some helpful classes which let us get more out of our training data.

- Imagine you train a neural network using only perfectly centered and upright photos
- If you then go and show that neural network an upside-down off center image, do you think it will perform well?

Of course not! Neural networks only recognize examples they see during training. So we need to introduce some imperfections into our training data so that our neural network can deal with un-ideal situations:

- `transforms.RandomResizeCrop` is a class which crops an image to a specific size, but with the image being off center
- `transforms.RandomHorizontalFlip` is a class which randomly mirrors an image

These transforms help introduce some unpredictability and imperfections into our training data. This will make our neural network more robust when we test it in the real world.

### Training and Validation Transforms
We will create 2 different transforms using the classes described above:

- `training_transforms`: Applied to all training data
  - This will make sure the images have the correct properties
  - And also introduce some variability (random crop, flips) so our neural network can handle imperfect data
- `validation_transforms`: Applied to input data when testing our neural network
  - This will make sure images have the correct properties
  - But no random variability (crops, flips) will be introduced
  - These will be real images that we want to test our neural network on
  - So we don't want to screw them up in any way

Below is the code for those 2 transforms.

In [ ]:
# The special average and standard deviation values which the MobileNet 2 dataset had
mobilenet_mean = [0.485, 0.456, 0.406]
mobilenet_std = [0.229, 0.224, 0.225]

# Transform we will apply to all our training data
training_transforms = transforms.Compose([
    # Introduce variability and imperfections into our data
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),

    # Ensure the average and standard deviation match MobileNet 2's original data
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mobilenet_mean,
        std=mobilenet_std,
    )
])

# Transform we will apply to any images we want to classify with our neural network
validation_transforms = transforms.Compose([
    # Ensure the correct size
    transforms.Resize(256),
    transforms.CenterCrop(224),

    # Ensure the average and standard deviation match MobileNet 2's original data
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mobilenet_mean,
        std=mobilenet_std,
    )
])

# This is a special transform which reverses the effects of changing the red, green, and blue pixel's average and standard deviation values
# This is needed because once we change those pixel values the image looks really weird to human eyes
# So we need to reverse this transformation to see what the image looks like
unnormalize_transform = transforms.Normalize(
    mean=[-m / s for m, s in zip(mobilenet_mean, mobilenet_std)],
    std=[1 / s for s in mobilenet_std]
)

Now that we have the transforms defined we need to apply the transforms to our data.

The easiest way to do this is make a custom PyTorch `Dataset` class which runs a transform on a dataset item.

We will use this custom `TransformDataset` later on.


In [ ]:
# Takes an existing dataset and transforms its items
class TransformDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # Get the original dataset item
        sample, label = self.dataset[idx]

        # Run the transform on the dataset item
        sample = self.transform(sample)

        # Return the transformed dataset item
        return sample, label

# Check In - Normalizing with PyTorch
We just learned how to normalize our images using PyTorch. Here are the key things you should know:

- PyTorch transforms help us edit images
  - We can combine multiple transforms together to perform multiple edits at once
  - We can ensure red, green, and blue pixel values are between 0.0 and 1.0 with transforms
  - We can ensure the average and standard deviation of an image match the original MobileNet 2 training data using transforms
- We introduce random variability in training data so our neural network can handle the imperfections of real life images better
- Images input into our classifier (not during training) must also have thse transforms applied to them because the training data had them so the model is used to images transformed this way
- We created a custom `TransformDataset` which will be used to apply these transforms to our training data

## Splitting into Training, Test, and Evaluation Datasets
Just like our other neural network we need to split our datasets into training, test, and evaluation datasets:

- **Training:** Used to update the weights and biases of the model
- **Test:** Used between each round of training to see how the training process is going
- **Evaluation:** Used at the very end to see how well our final model works

We will be using a 60% training, 20% test, and 20% validation split.

- Previously we used PyTorch's `random_split` function to split up our dataset
- However, since each class might have a different number of images we want to be careful and ensure each class is split up proportionally.

This requires a bit of clever code which is written below. Don't worry too much about understanding it. Just know that it ensures that 60% of images from each class are used for training, 20% for testing, and 20% for evaluation.

In [ ]:
# For each image class get the dataset item indexes
class_idxs = defaultdict(list)
for sample_idx, label in enumerate([ sample[1] for sample in dataset.samples ]):
    class_idxs[label].append(sample_idx)

# Shuffle the indexes of each item for each class
for label, indexes in class_idxs.items():
    class_idxs[label] = np.random.permutation(indexes).tolist()

# Create dataset subsets
training_pct = 0.6
test_pct = 0.2
eval_pct = 0.2

training_idxs = []
test_idxs = []
eval_idxs = []

for label, indexes in class_idxs.items():
    training_size = int(len(indexes) * training_pct)
    test_size = int(len(indexes) * test_pct)
    eval_size = len(indexes) - (training_size + test_size)

    training_idxs.extend(indexes[:training_size])
    test_idxs.extend(indexes[training_size:training_size + test_size])
    eval_idxs.extend(indexes[training_size + test_size:])

Now that we have randomly split our training data by those proportions we can use our `TransformDataset` class from earlier:


In [ ]:
training_dataset = Subset(TransformDataset(dataset, training_transforms), training_idxs)
test_dataset = Subset(TransformDataset(dataset, validation_transforms), test_idxs)
eval_dataset = Subset(TransformDataset(dataset, validation_transforms), eval_idxs)

`training_dataset`, `test_dataset`, and `eval_dataset` now each all contain a portion of the images according to our 60%, 20%, 20% split.

## Shuffling Data Order During Fine Tuning
Introducing variability during the fine tuning process helps our neural network work better in the real world. We already saw this with our random cropping and flipping of training images.

We can do 2 more things in a similar vein during the fine tuning process:

- Shuffle the order in which images are shown to the model
  - This helps ensure that our model's weights and biases are updated fairly based on all images, not just the first few images
- Show more than one image to the model before updating weights and biases
  - We just compute the error for each image and add it up
  - More efficient, updating weights and biases takes some time to complete
  - Seeing more images before updating weights and biases helps create a greater error comprised of more variaty

Both of these practices are commonly used when training and fine tuning convolutional neural networks.

To do this we can use PyTorch's `DataLoader` class:

- The `shuffle` argument indicates if data should be shuffled
- The `batch_size` argument indicates how many images should be shown to the model before weights and biases are updated
- The `DataLoader` class also helps retrieve our images from our folders during training


In [ ]:
# Configure training data to shuffled
training_loader = DataLoader(training_dataset, batch_size=4, shuffle=True)

# Do not shuffle
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
eval_loader = DataLoader(eval_dataset, batch_size=4, shuffle=False)

## Look at Our Training Data
Okay now that we have dataset all split up, let's take a look at some of the images.

The code below shows 5 random images from your training data:

In [ ]:
def imshow(inp, title = None):
    # Un-normalize image pixel data
    inp = unnormalize_transform(inp)

    # Flip pixel channels into correct value for displaying
    inp = inp.numpy().transpose((1, 2, 0))

    # Ensure values aren't too big or small to display
    inp = np.clip(inp, 0, 1)

    # Show image
    display(transforms.ToPILImage()(inp))

    # Show title
    if title is not None:
        display(title)

num_to_show = 5
num_shown = 0
for batch in training_loader:
    for img, label in zip(*batch):
       imshow(img, dataset.classes[label])
       num_shown += 1

       if num_shown >= num_to_show:
          break
    if num_shown >= num_to_show:
          break

# Check In - Splitting Data
We just learned how to split training, test, and evaluation data in PyTorch for images. Here are the key points you should understand:

- We are splitting our data so 60% is used for training, 20% is used for testing, and 20% and used for evaluation
- To improve the training process we are also:
  - Shuffling the order in which images are shown to the neural networking during training
  - Showing the neural network 4 images at a time before updating weights and biases
  - These practices help ensure the fine tuning process works well

# Setting up our Neural Network
Now that we have all training data completely set up it is time to move on to our neural network.

You may recall from earlier that the MobileNet 2 neural network is extremely complex:

- 53 layers
- 3.4 million parameters

With our weather neural network earlier we manually defined each layer. However, that won't work for 2 reasons:

- The structure of MobileNet 2 is very specific so that it learns image categories and works well on less powerful computers
- Since we are fine tuning MobileNet 2 we need those 3.4 million parameters (weights and biases)
  - These are the values which are the result of 3 days of training on very expensive computers

Luckily PyTorch once again helps us out here. PyTorch comes built in with a bunch of commonly used neural networks. MobileNet 2 being one of them.

- All we have to do is call the `models.mobilenet_v2` function and PyTorch will build the MobileNet v2 neural network for us
- Providing the `pretrained=True` argument tells PyTorch to also download those 3.4 million weights and biases which are a result of 3 days of training

In [ ]:
model = models.mobilenet_v2(pretrained=True)

Now our `model` variable contains the MobileNet 2 neural network.

The only problem is, right now our neural network is set up to classify between 1,000 different objects. And we're trying to fine tune the model instead.

## Swapping the Output Layer
To fine tune a neural network we leave all the input neurons, and hidden layer neurons in place.

We then take the final output layer and swap out its neurons with our own.

- We want one neuron per class we are going to fine tune for
- Each neuron will output a value which is the probability that the image contains that class

So if we want to fine tune MobileNet 2 to classify llamas and ducks we would swap out the output layer for 2 neurons. One for llamas being in the image and one for ducks being in the image.

The MobileNet 2 neural network PyTorch gives us stores its output layer in the `model.classifier[1]` variable (A class variable named `classifier` which is a list, and we are accessing the **second** item in that list).

We can go and replace this with our own output layer. But before we do so, let's take a look at what it is:

In [ ]:
model.classifier[1]

You can see that:

- The output layer is a `nn.Linear` class (just like our weather neural network)!
- The output layer MobileNet 2 comes with has 1,000 output neurons (called "features" by PyTorch)
- The layer before the output layer has 1,280 hidden neurons (again, called "features")

Now let's replace the output layer. We want to replace it with our own `nn.Linear(in, out)` class.

- The first argument (`in`) will be the number of neurons from the hidden layer before the output layer
  - We can get this by accessing the `model.classifier[1].in_features` variable
- The second argument (`out`) will be the number of output neurons
  - This should be the number of classes our dataset has
  - Remember the `dataset.classes` variable from above?
  - It's a list of all the class names in our dataset
  - To get the number of classes we just get the length of this list

In [ ]:
# Number of neurons in hidden layer before output layer
previous_layer_neurons = model.classifier[1].in_features

# Classes in our dataset
num_classes = len(dataset.classes)

print("Number of neurons in previous layer: ", previous_layer_neurons)
print("Dataset classes=", dataset.classes)
print("Number of classes: ", num_classes)

# Replace final output layer with our own
model.classifier[1] = nn.Linear(previous_layer_neurons, num_classes)

And just like that, our Mobile Net 2 neural network is ready for fine tuning.

# Check In - Neural Network Setup
We just learned how to create a MobileNet 2 neural network and set it up for fine tuning. Here are the points you should understand:

- PyTorch comes with MobileNet 2's neural network and weights + parameters built in
  - Just called `models.mobilenet_v2(pretrained=True)` to get it
- To fine tune a neural network we swap its output layer with our own
  - We did this by assigning a new value to `model.classifier[1]`

# Note
Below you will see some red text that says "RuntimeError: This error is on purpose...".

Ignore this and move on with the lesson. This code just prevents the model fine tuning code below from running whenever you click the "Run all" button in Colab.

In [ ]:
raise RuntimeError("This error is on purpose, we just want to make sure you don't run any code below this line")

# Fine Tuning
Now we have a dataset and a neural network which is ready for fine tuning. Now we just have to complete the process of fine tuning.

Fine tuning is very similar to training a neural network. The only difference is:

- In training we updating all the weights and biases in all layers of a neural network
- In fine tuning we only update the weights and biases of the output layer of our neural network

You will recall from training our weather neural network that in PyTorch we use an "optimizer" to update the weights and biases.

- We provide this optimizer with the weights and biases we want it to update
- So to fine tune we just provide the optimizer with only the weights and biases from the output layer

In the code below you can see we provide the optimizer only with the parameters from our `model.classifier[1]` output layer.

Other than that the code is almost identical to the weather neural network code:

- We have a `learning_rate` variable which tunes how much the optimizer updates our weights and biases each training loop
- We have an `error_fn` which tells the training process how close the neural network's prediction is to the correct result
  - In our weather neural network we used Mean Squared Error
  - Here we will use "Cross Entropy Loss"
  - This sounds like a crazy name, but it basically is just a slightly different version of Mean Squared Error
  - It is better for these sorts of classification models
- We show the model some images and see which class it predicts those images will be in
- We calculate the error based on how close the model's predictions were
- The optimizer updates the model's weights and biases
- Repeat

# Check In - Fine Tuning
We just learned how to fine tune a neural networking in PyTorch. Here are the key points you should know:

- The only difference between fine tuning and training a model is which layers we update the weights and biases of
  - Fine tuning: Only update output layer
  - Training: Update all layers
- In PyTorch an optimizer updates weights and biases during training
  - To fine tune a neural network we tell the optimizer to only update the output layer

## Running Fine Tuning
Below is the fine tuning code. Click the run button to fine tune MobileNet 2 based on your own classes and training data.

Wait for the code to finish fine tuning (This should take about 2-5 minutes).

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
print("Loading neural networking, please wait... (this count take a minute or so)")

# Setup the neural network
model = model.to(device)

# Define how we're going to train the model
learning_rate = 0.001
num_training_rounds = 50

error_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.classifier[1].parameters(), lr=learning_rate, momentum=0.9)

stop_training_below_error = 0.05

def training_loop(model, loader, optimizer, error_fn):
    # Function which runs our model through a dataset and returns the error
    total_error = 0.0

    for batch_input, batch_labels in loader:
        # See what the model predicts
        prediction = model(batch_input.to(device))

        # Calculate the error (how close the prediction was to the correct result)
        error = error_fn(prediction, batch_labels.to(device))

        total_error = total_error + error

        # If an optimizer is specified then updates the model parameters using it
        if optimizer is not None:
            # Get the derivative of all the weights and biases in relation to the error equation
            error.backward()

            # Update the weights and biases by a small amount in the correct direction
            optimizer.step()

            # Reset the derivatives to zero so we can re-calculate them for the next dataset item
            optimizer.zero_grad()

    return total_error


# Training loop
for training_round in range(num_training_rounds):
    # Update the parameters of our model with the training dataset
    # Tell our model it's about to be trained
    model.train()

    training_loop(
        model=model,
        loader=training_loader,
        optimizer=optimizer,
        error_fn=error_fn,
      )

    # Use the test dataset to determine how training is going
    # Tell our model it's about to be evaluated
    model.eval()

    test_error = training_loop(
        model=model,
        loader=test_loader,
        optimizer=None, # Do not provide an optimizer, so parameters are not updated
        error_fn=error_fn,
      )

    # Stop training if our error is low enough
    if test_error <= stop_training_below_error:
        print(f"In training round {training_round} we achieved test error {test_error}, exiting early")
        break

    # Print our progress every training round
    print(f"Training round {training_round}, test error {test_error}")

# Finally see how accurate our model is via the evaluation dataset
# Tell our model it's about to be evaluated
model.eval()

eval_error = training_loop(
    model=model,
    loader=eval_loader,
    optimizer=None, # Do not provide an optimizer, so parameters are not updated
    error_fn=error_fn,
)

print(f"After {training_round+1} training rounds the model has an evaluation error of {eval_error}")

## Making Sense of Our Model Output
Alright, now you have a version of MobileNet 2 which is specially fine tuned to recognize the classes you set up.

During training you should have seen the error go down with each training round.

So how do we use our fine tuned neural network, and what does its output look like? To find out let's ask our neural network to tell us what object is in an image.

_Run the code below_

In [ ]:
# RUN ME

# Get one image from our evaluation dataset
eval_img, eval_class = eval_dataset[0]

# The eval_class value is just a number
# This number is an index in the dataset.classes list
# To get the name of the class we can access the index in that list
class_name = dataset.classes[eval_class]

# Show the evaluation image and its class name
imshow(eval_img, "Eval image showing a " + class_name)

# Prepare our image to be put into our neural network
# The MobileNet 2 neural network is actually setup to classify multiple images at a time
# So it expects a list of images
# To create this list we use a special PyTorch function called "unsqueeze" which makes a special single item list (idx why PyTorch named it that)
# .to(device) sends the image over to the GPU so we can process our image quickly
input_batch = eval_img.unsqueeze(0).to(device)

# Ask our neural network what is in the image
prediction = model(input_batch)

# Since MobileNet 2 is meant to process multiple images at a time it will also return a list of results
# So we call "squeeze" to get a single item out of the list (The reverse of what we did when inputting the image)
print("Prediction is:")
output = prediction.squeeze(0).tolist()
output

Alright, so we can see the image we input into the neural network. And we can see that the output of the neural network is... a list of numbers?

Remember, our neural network outputs the probability that the image contains each classes. You may notice that the output list is the same length as our `dataset.classes` list:

_Run the code below_

In [ ]:
# RUN ME

print("Neural network output:")
print(output)

print("Dataset classes:")
print(dataset.classes)

The first number in our neural network output corresponds to the probability that the image contains our first class. And the second output of our neural network corresponds to the probability that the image contains our first class. Ect.

We can loop through both the output and the class names to show this:

_Run the code below_

In [ ]:
# RUN ME

for index in range(len(dataset.classes)):
    print("Class " + dataset.classes[index])
    print("  Probability ", output[index])

You may notice though, that these probability numbers look kind of weird. For example when writing this lesson I made a neural network that categorized between cats and trains. And here were my probabilities:

![Cats and trains probabilities, looking weird](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/cats-and-trains-probabilities-before-softmax.png?raw=true)

- We learned earlier that probabilities should be between 0.0 (aka 0%) and 1.0 (1%)
- We also know that probabilities should add up to 100%
- So why does our model say there is a 3.32 (aka 332%) probability that the image contains a cat and a -3.38 (aka -338%) probability that the image contains a train?
- And why do these numbers not add up to 100%?
- Clearly these numbers are trying to indicate that the image definitely contains a cat and does not contain a train

This is a very common thing with output neurons in neural networks. We will have numbers that are conveying a probability like value, but they won't be between 0.0 and 1.0.

- This is actually very easy to fix
- We just need to normalize the values of the output list
- A special function used in machine learning named "softmax" does this for us

The "softmax" function takes a list of input values and converts it so that:

- All the values are between 0.0 and 1.0
- All values add up to 1.0
- Preserves the relationship between numbers
  - So in the screenshot above the cats probability will be much larger than the trains probability

To use "softmax" in PyTorch we just use the `nn.functional.softmax` function:

_Run the code below_

In [ ]:
# RUN ME
output_softmax = nn.functional.softmax(torch.tensor(output), dim=0)
output_softmax.tolist()

Now you can see the output makes a lot more sense. Probabilities are all between 0.0 an 1.0. And the probabilities add up to 1.0 (100%).

So how can we use this output? Well typically we want to figure out which class the neural network thought was most likely in the image.

In PyTorch classes are just numbers which are indexes of the `dataset.classes` list. And the `datasets.classes` list contains names of classes.

- So if an image is class `0` that means it is the first class in the `datasets.classes` list

We can use the `argmax()` function on a tensor in PyTorch to get the index with the biggest value. Then we can look up the name of that class in `dataset.classes`:

_Run the code below_

In [ ]:
# RUN ME
class_index = output_softmax.argmax().item()
class_name = dataset.classes[class_index]

print("Image was class", class_index)
print("The name of that class was", class_name)

# Check In - Making Sense of our Model Output
We just learned how to interpret and work with the output of our fine tuned MobileNet 2 neural network. Here are the key points you should know:

- Our neural network outputs a list of values indicating how likely it is that the image contains each of our classes
  - Larger values = More likely to contain our class
  - Smaller values = Less likely to contain our class
- The order of our neural network outputs list is the same order as our `datasets.classes` list
  - The first item in the output list is the likelyhood that our image contains the first class
  - The second item in the output list is the likelyhood that our image contains the second class
  - And so on
- Output values need to be processed with the softmax function to conver them into probabilities
- We can get the index of the class with the highest probability via the `argmax` PyTorch function

## Evaluating Our Fine Tuned Neural Network
Let's put together everything we learned above to see what class our neural network thinks is in each of our evaluation images.

In [ ]:
# Variables for tallying up the results
correct = 0
wrong = 0
num_shown = 0

# For each batch of images in the evaluation dataset
for batch in eval_loader:
    # For each image
    for img, class_ in zip(*batch):
        # We give the image to the neural network
        prediction = model(img.unsqueeze(0).to(device))
        probabilities = nn.functional.softmax(prediction.squeeze(0), dim=0)

        # Get the predicted class
        predicted_class = prediction.argmax().item()
        predicted_label = dataset.classes[predicted_class]
        prediction_probability = probabilities[predicted_class]

        actual_label = dataset.classes[class_]

        # Tally if we are correct or not
        if predicted_class != class_:
            wrong += 1
        else:
            correct += 1

        # Show image
        title = f"^^^ Predicted: {predicted_label} ({prediction_probability.item():0.2f}), Actual: {actual_label} ^^^"
        imshow(img, title)
        num_shown += 1

# Show totals
print(f"correct={correct}, wrong={wrong}")

In [ ]:
# Quantize model
model.qconfig = torch.quantization.get_default_qconfig('qnnpack')
quant_model = torch.quantization.prepare(model.to("cpu"), inplace=False)
quant_model = torch.quantization.convert(quant_model, inplace=False)

In [ ]:
torch.save(model.state_dict(), f"{datasets_parent}/model.pt")

In [ ]:
with open(f"{datasets_parent}/classes.json", "w") as f:
  json.dump(dataset.classes, f)

In [ ]:
torch.__version__

In [ ]:
list(model.state_dict())

In [ ]:
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

In [ ]:
sample_inputs = (training_dataset[0][0].unsqueeze(0),)

et_program = to_edge_transform_and_lower(
    torch.export.export(model.to("cpu"), sample_inputs),
    partitioner=[XnnpackPartitioner()]
).to_executorch()

with open(f"{datasets_parent}/model.pte", "wb") as f:
    f.write(et_program.buffer)

In [ ]:
import pickle

with open(f"{datasets_parent}/eval_dataset.pkl", "wb") as f:
    pickle.dump(list(eval_dataset), f)